# 🎬 AutoVSF - Google Colab Edition

Automated hard subtitle (hardsub) extraction tool using VideoSubFinder and Google Drive OCR.

🔗 **Repository:** [https://github.com/lionc2240/autovsf-colab](https://github.com/lionc2240/autovsf-colab)

## 🚀 Step 1: Connect to Google Drive
We will save all code and results on Drive to prevent data loss when Colab shuts down.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create working directory on Drive
WORKDIR = "/content/drive/MyDrive/AutoVSF"
if not os.path.exists(WORKDIR):
    os.makedirs(WORKDIR)
    print(f"✅ Working directory created at: {WORKDIR}")

%cd $WORKDIR

## 🛠️ Step 2: Environment Setup
Automatically download source code and install necessary system libraries (VideoSubFinder, Xvfb, ffmpeg...)

In [ ]:
# 1. Clone or update source code
import os
if os.path.exists("headless.py"):
    print("✅ Already in repository.")
    !git pull
elif os.path.exists("autovsf-colab"):
    %cd autovsf-colab
    print("✅ Repository found. Updating...")
    !git pull
else:
    print("📥 Cloning repository...")
    !git clone https://github.com/lionc2240/autovsf-colab.git
    %cd autovsf-colab

# 2. Run system setup script (Handles legacy libraries for Colab Ubuntu 22.04)
# Note: install.sh will automatically handle missing .deb files
!chmod +x install.sh
!./install.sh

## 🔑 Step 3: Google Cloud Configuration (OCR)
1. Upload your `credentials.json` file to the `AutoVSF/autovsf-colab/` folder on Google Drive.
2. If this is the first run, the tool will ask you to authenticate via a link.

In [ ]:
if not os.path.exists("credentials.json"):
    print("❌ ERROR: 'credentials.json' file not found!")
    print("👉 Please upload this file to the AutoVSF/autovsf-colab/ folder on your Drive and run this cell again.")
else:
    print("✅ credentials.json found. Ready!")

## 🎬 Step 4: Launch Subtitle Extraction
Choose your preferred extraction mode. Interactive (Step 4.1) is recommended for most users.

## 🎨 Step 4.1: Advanced Visual Mode (Interactive Cropping) (Recommended)
Interactive UI using Gradio Image Editor for precise subtitle area selection.

In [ ]:
import gradio as gr
import cv2
import numpy as np
import headless
import os
import config as C
import shutil
import time
from pathlib import Path
import subprocess

def get_sample_frame(video_path, timestamp=5):
    if not video_path or not os.path.exists(video_path):
        return None
    sample_frame_path = "/tmp/sample_frame.jpg"
    
    # Convert seconds to HH:MM:SS format for ffmpeg
    time_str = time.strftime('%H:%M:%S', time.gmtime(timestamp))
    
    try:
        subprocess.run([
            "ffmpeg", "-y", "-ss", time_str, "-i", video_path, 
            "-vframes", "1", "-q:v", "2", sample_frame_path
        ], check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print(f"Error extracting frame: {e.stderr.decode()}")
        # Fallback to 0s if timestamp fails
        subprocess.run([
            "ffmpeg", "-y", "-i", video_path, 
            "-vframes", "1", "-q:v", "2", sample_frame_path
        ], check=True)
        
    return sample_frame_path

def handle_upload(file):
    if file is None: return ""
    dest = os.path.join("/content", os.path.basename(file.name))
    shutil.copy(file.name, dest)
    return dest

def save_to_drive(srt_path, drive_folder):
    if not srt_path or not os.path.exists(srt_path):
        return "❌ SRT file not found. Please run extraction first."
    try:
        if not os.path.exists(drive_folder):
            os.makedirs(drive_folder, exist_ok=True)
        dest = os.path.join(drive_folder, os.path.basename(srt_path))
        shutil.copy(srt_path, dest)
        return f"✅ Saved to: {dest}"
    except Exception as e:
        return f"❌ Error saving to Drive: {str(e)}"

def start_extraction_advanced(video_path, image_editor, progress=gr.Progress()):
    if not video_path or not os.path.exists(video_path):
        yield "❌ Invalid video path.", None
        return
    if image_editor is None or "composite" not in image_editor:
        yield "❌ Please click 'Get Sample Frame' and crop the subtitle area.", None
        return

    try:
        yield "⏳ Analyzing original sample frame...", None
        orig_frame_path = "/tmp/sample_frame.jpg"
        if not os.path.exists(orig_frame_path):
            yield "❌ Original sample frame not found. Please try again.", None
            return
            
        orig_img = cv2.imread(orig_frame_path)
        orig_h, orig_w, _ = orig_img.shape

        cropped_img = image_editor["composite"] 
        h, w, c = cropped_img.shape

        x_min, y_min, x_max, y_max = 0, 0, w, h
        is_cropped = False

        if h < orig_h or w < orig_w:
            is_cropped = True
            if c == 4:
                cropped_bgr = cv2.cvtColor(cropped_img, cv2.COLOR_RGBA2BGR)
            else:
                cropped_bgr = cropped_img
            
            res = cv2.matchTemplate(orig_img, cropped_bgr, cv2.TM_CCOEFF_NORMED)
            _, _, _, max_loc = cv2.minMaxLoc(res)
            x_min, y_min = max_loc
            x_max = x_min + w
            y_max = y_min + h

        elif c == 4:
            alpha = cropped_img[:, :, 3]
            if np.any(alpha < 250):
                non_transparent = alpha > 10
                if np.any(non_transparent):
                    rows = np.any(non_transparent, axis=1)
                    cols = np.any(non_transparent, axis=0)
                    y_indices = np.where(rows)[0]
                    x_indices = np.where(cols)[0]
                    if len(y_indices) > 0 and len(x_indices) > 0:
                        y_min, y_max = y_indices[0], y_indices[-1]
                        x_min, x_max = x_indices[0], x_indices[-1]
                        if (x_max - x_min < orig_w - 5) or (y_max - y_min < orig_h - 5):
                            is_cropped = True

        if not is_cropped:
            yield "⚠️ You haven't cropped yet. Please use the crop tool on the image editor.", None
            return

        top_crop = (orig_h - y_min) / orig_h
        bottom_crop = (orig_h - y_max) / orig_h
        left_crop = x_min / orig_w
        right_crop = x_max / orig_w

        print(f" LOGIC MAPPED VSF COORDINATES -> Top: {top_crop:.4f}, Bottom: {bottom_crop:.4f}, Left: {left_crop:.4f}, Right: {right_crop:.4f}")
        yield "⏳ Preparing VideoSubFinder...", None

        C.state.reset()
        
        def run_task():
            try:
                y_min_vsf, y_max_vsf = min(top_crop, bottom_crop), max(top_crop, bottom_crop)
                if y_min_vsf == y_max_vsf:
                    y_max_vsf += 0.05
                headless.run_headless(video_path, y_max_vsf, y_min_vsf, left_crop, right_crop)
            except Exception as e:
                print(f"Error in headless thread: {e}")

        import threading
        t = threading.Thread(target=run_task, daemon=True)
        t.start()

        fake_prog = 0.0
        progress(0, desc="🚀 Initializing VideoSubFinder...")
        while t.is_alive():
            video_name = Path(video_path).stem
            output_dir = os.path.join("/content", f"{video_name}_out", "RGBImages")
            if not os.path.exists(output_dir):
                output_dir = os.path.join(str(Path(video_path).parent), f"{video_name}_out", "RGBImages")
            
            img_count = 0
            if os.path.exists(output_dir):
                img_count = len([f for f in os.listdir(output_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])

            if C.state.total > 0:
                ocr_prog = C.state.done / C.state.total
                prog = 0.85 + (ocr_prog * 0.15)
                desc_str = f"⚡ Phase 2/2: Running OCR on Drive... Progress: {C.state.done}/{C.state.total} frames."
                progress(prog, desc=desc_str)
                yield f"⚙️ Running OCR (Please wait)... Progress: {C.state.done}/{C.state.total} ({prog*100:.1f}%)", None
            else:
                fake_prog = min(fake_prog + 0.015, 0.85)
                desc_str = f"🎞️ Phase 1/2: Scanning video for subtitles... Found {img_count} frames."
                progress(fake_prog, desc=desc_str)
                yield f"⏳ Scanning video stream (Please wait)... Extracted {img_count} frames ({fake_prog*100:.1f}%)", None
            
            time.sleep(1.5)

        srt_out = str(Path(video_path).with_suffix('.srt'))
        if os.path.exists(srt_out):
            yield f"✅ Extraction successful! File: {srt_out}", srt_out
        else:
            yield "⚠️ Finished. Check logs if .srt file is missing.", None

    except Exception as e:
        yield f"❌ Error processing image data: {str(e)}", None

with gr.Blocks(title="AutoVSF Advanced") as demo:
    gr.Markdown("# 🎨 AutoVSF Advanced Visual Mode (Recommended)")
    
    with gr.Row():
        with gr.Column():
            video_in = gr.Textbox(label="Video Path", value="/content/video_test.mp4")
            with gr.Row():
                upload_btn = gr.UploadButton("📁 Upload Video", file_types=["video"])
                timestamp_in = gr.Number(label="Sample Frame Timestamp (sec)", value=5, precision=0)
            btn_frame = gr.Button("📷 Get Sample Frame", variant="secondary")
            
        with gr.Column():
            img_editor = gr.ImageEditor(
                label="1. Click Crop icon -> 2. Select subtitle area -> 3. Click 'Apply' (check mark)", 
                type="numpy", 
                crop_size=None,
                transforms=["crop"],
                brush=False,
                eraser=False
            )
            
    btn_start = gr.Button("🚀 Start Extraction", variant="primary")
    st_out = gr.Textbox(label="System Status")
    srt_file = gr.File(label="Download SRT Result", interactive=False)
    
    with gr.Accordion("Google Drive Options", open=False):
        drive_path = gr.Textbox(label="Drive Folder Path", value="/content/drive/MyDrive/AutoVSF/Results")
        btn_drive = gr.Button("💾 Save SRT to Drive")
        drive_status = gr.Markdown()

    upload_btn.upload(handle_upload, upload_btn, video_in)
    btn_frame.click(get_sample_frame, [video_in, timestamp_in], [img_editor])
    btn_start.click(start_extraction_advanced, [video_in, img_editor], [st_out, srt_file])
    btn_drive.click(save_to_drive, [srt_file, drive_path], drive_status)

demo.queue().launch(share=True)


## 🚀 Step 4.2: CLI Mode (Fast Execution - Auto 30%)
Quickly run extraction using relative coordinates. Ideal for batches or known subtitle positions.

In [ ]:
# @title 🚀 Step 4.2: CLI Mode (Fast Execution - Auto 30%)
# @markdown ### 💡 Crop Region Selection Guide:
# @markdown - Parameters range from **0.0** to **1.0** (0% - 100% of video dimensions).
# @markdown - **Y Axis (Vertical):** **0.0 is Bottom**, **1.0 is Top**.
# @markdown   - *Note:* To scan the bottom 30% of the video (common subtitle area), set **Bottom = 0.0** and **Top = 0.3**.
# @markdown - **X Axis (Horizontal):** 0.0 is Left, 1.0 is Right.

VIDEO_PATH = "/content/video_test.mp4" # @param {type:"string"}
TOP_CROP = 0.3 # @param {type:"number"}
BOTTOM_CROP = 0.0 # @param {type:"number"}
LEFT_CROP = 0.0 # @param {type:"number"}
RIGHT_CROP = 1.0 # @param {type:"number"}

!python3 headless.py "$VIDEO_PATH" --top $TOP_CROP --bottom $BOTTOM_CROP --left $LEFT_CROP --right $RIGHT_CROP

## 🎨 Step 4.3: Basic Visual Mode (Slider Selection)
Simple interface using sliders to define the extraction boundaries.

In [ ]:
# @title 🎨 Step 4.3: Basic Visual Mode (Slider Selection)
try:
    import gradio as gr
except:
    !pip install -q gradio
    import gradio as gr

import os
import threading
import time
from pathlib import Path
import config as C
import headless

def load_last_settings():
    cfg = C.load()
    profile = cfg.get("crop_profiles", {}).get("colab_last_run", {})
    return (
        profile.get("top", 0.3), 
        profile.get("bottom", 0.0), 
        profile.get("left", 0.0), 
        profile.get("right", 1.0)
    )

def save_settings(top, bottom, left, right):
    cfg = C.load()
    if "crop_profiles" not in cfg: cfg["crop_profiles"] = {}
    cfg["crop_profiles"]["colab_last_run"] = {
        "top": top, "bottom": bottom, "left": left, "right": right
    }
    C.save(cfg)
    return "✅ Profile saved to settings.json"

def start_extraction(video_path, top, bottom, left, right, progress=gr.Progress()):
    if not video_path or not os.path.exists(video_path):
        return "❌ Video path invalid."
    
    save_settings(top, bottom, left, right)
    C.state.reset()
    
    def run_task():
        try:
            y_min, y_max = min(top, bottom), max(top, bottom)
            if y_min == y_max: y_max += 0.1
            headless.run_headless(video_path, y_max, y_min, left, right)
        except Exception as e: print(f"Error: {e}")
    
    t = threading.Thread(target=run_task, daemon=True)
    t.start()
    
    progress(0, desc="🚀 Initializing...")
    while t.is_alive():
        if C.state.total > 0:
            prog = C.state.done / C.state.total
            progress(prog, desc=f"⏳ OCR Progress: {C.state.done}/{C.state.total}")
        else:
            progress(0, desc="🎞️ Scanning video...")
        time.sleep(1)

    if C.state.total > 0 and C.state.done >= C.state.total:
        return f"✅ Done! Output: {Path(video_path).with_suffix('.srt')}"
    return "⚠️ Finished. Check logs if file is missing."

def get_status_text():
    if C.state.total == 0: return "### Status: Ready"
    prog = C.state.done / C.state.total if C.state.total > 0 else 0
    txt = f"### Progress: {C.state.done}/{C.state.total} ({prog*100:.1f}%)"
    if C.state.done >= C.state.total: txt += "\n✅ Done!"
    return txt

with gr.Blocks(title="AutoVSF Basic") as demo:
    gr.Markdown("# 🎬 AutoVSF Basic Interface (Slider Selection)")
    gr.Markdown("**Coordinate System:** 0.0 = Bottom, 1.0 = Top. Adjust sliders to select subtitle area.")
    
    with gr.Row():
        with gr.Column(scale=2):
            video_in = gr.Textbox(label="Video Path", value="/content/video_test.mp4")
            with gr.Group():
                gr.Markdown("### ✂️ Crop Settings (Relative 0.0 - 1.0)")
                with gr.Row():
                    t_sld = gr.Slider(0, 1, value=0.3, step=0.01, label="Top (Upper boundary)")
                    b_sld = gr.Slider(0, 1, value=0.0, step=0.01, label="Bottom (Lower boundary)")
                with gr.Row():
                    l_sld = gr.Slider(0, 1, value=0.0, step=0.01, label="Left")
                    r_sld = gr.Slider(0, 1, value=1.0, step=0.01, label="Right")
            with gr.Row():
                btn_save = gr.Button("💾 Save Profile")
                btn_start = gr.Button("🚀 Start Extraction", variant="primary")
        
        with gr.Column(scale=1):
            st_out = gr.Textbox(label="System Status", interactive=False)
            prog_txt = gr.Markdown("### Status: Ready")
            
            demo.load(load_last_settings, None, [t_sld, b_sld, l_sld, r_sld])
            
            if hasattr(gr, "Timer"):
                gr.Timer(2).tick(get_status_text, None, prog_txt)
            else:
                demo.load(get_status_text, None, prog_txt)
            
    btn_save.click(save_settings, [t_sld, b_sld, l_sld, r_sld], st_out)
    btn_start.click(start_extraction, [video_in, t_sld, b_sld, l_sld, r_sld], st_out)

demo.queue().launch(share=True, height=600)


## 🧹 Step 5: Cleanup (Optional)
Temporary images are stored in `/content` and will be auto-deleted when session ends.

In [ ]:
import os
from pathlib import Path

out_dir = f"/content/{Path(VIDEO_PATH).stem}_out"
if os.path.exists(out_dir):
    print(f"🧹 Removing temporary folder: {out_dir}")
    !rm -rf "$out_dir"
    print("✅ Cleanup complete.")